# Stage 3: cross-fitted residual sequence ensemble

This notebook is self-contained. It mounts Google Drive, checks out the repository, installs the locked environment, copies the competition data, creates Stage 2 automatically if it is missing, and then trains the promoted two-seed Stage 3 residual ensemble. With an existing Stage 2 artifact, allow roughly 3--10 minutes on a CPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)

drive_data = drive_root / 'data'
assert (drive_data / 'train').is_dir(), f'Missing train directory: {drive_data / "train"}'
if not (data_dir / 'train').is_dir():
    shutil.copytree(drive_data, data_dir, dirs_exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_dir)
print('data:', data_dir)
print('artifacts:', artifact_dir)

In [ ]:
STAGE2_RUN_ID = 'stage2_pf_trend_blend_full_v001'
stage2_dir = artifact_dir / STAGE2_RUN_ID
if not (stage2_dir / 'metrics.json').is_file():
    assert not stage2_dir.exists() or not any(stage2_dir.iterdir()), (
        f'Incomplete Stage 2 artifact exists: {stage2_dir}. Use a new run ID or remove it manually.'
    )
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/pf_trend_blend.yaml',
        '--run-id', STAGE2_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
stage2_metrics = json.loads((stage2_dir / 'metrics.json').read_text())
print('Stage 2 RMSE:', stage2_metrics['pooled_rmse'])

In [ ]:
SMOKE_RUN_ID = 'stage3_residual_hgb_smoke_v001'
smoke_dir = artifact_dir / SMOKE_RUN_ID
if not (smoke_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-residual',
        '--config', 'configs/experiment/stage3_residual_hgb.yaml',
        '--base-run', str(stage2_dir),
        '--run-id', SMOKE_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
json.loads((smoke_dir / 'metrics.json').read_text())

In [ ]:
RUN_ID = 'stage3_residual_hgb_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-residual',
        '--config', 'configs/experiment/stage3_residual_hgb.yaml',
        '--base-run', str(stage2_dir),
        '--run-id', RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
stage3_metrics = json.loads((run_dir / 'metrics.json').read_text())
stage3_metrics

In [ ]:
{
    'stage2_rmse': stage2_metrics['pooled_rmse'],
    'stage3_rmse': stage3_metrics['pooled_rmse'],
    'rmse_delta': stage3_metrics['pooled_rmse'] - stage2_metrics['pooled_rmse'],
}

In [ ]:
comparison_dir = artifact_dir / 'stage3_vs_stage2_v001'
if not (comparison_dir / 'comparison.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-compare',
        '--baseline', str(stage2_dir),
        '--candidate', str(run_dir),
        '--output-dir', str(comparison_dir),
    ], cwd=repo_dir, check=True)
json.loads((comparison_dir / 'comparison.json').read_text())